# 05 — Multiple Runs: Clustering Stability Across Seeds

Assess how sensitive k-means clustering results are to random initialisation
by repeating each clustering condition 10 times with different seeds.

**Four clustering conditions:**
1. **Federated k-means on federated-corrected data** — real FeatureCloud Docker runs; seed injected via `config_kmeans.yml` (`algorithm.seed`)
2. **Federated k-means on uncorrected data** — same approach
3. **Central k-means on centrally-corrected data** — `run_central_kmeans` with explicit seed
4. **Central k-means on uncorrected data** — same

**Federated runs:**  
The `fc_kmeans_upd` app reads `seed` from `algorithm.seed` in `config_kmeans.yml`.
For each seed the per-site config files are rewritten, one FeatureCloud Docker test is
launched (10 seeds × 4 active datasets × 2 variants = 80 tests), results are read immediately,
and the next seed begins.

**Prerequisites:** Docker running + FeatureCloud controller at `CONTROLLER_HOST`.
Run notebooks 01–03 first (prepared matrices must exist in `<dataset>/prepared/`).

**Evaluation:** ARI against condition labels only, reported as boxplots over 10 seeds.


## Imports

In [ ]:
import sys
from pathlib import Path
from typing import Dict, List, Sequence

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import adjusted_rand_score

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation_utils.real_datasets_utils import (
    dataset_configs,
    load_feature_matrix,
    load_matrix_with_lfs_fallback,
    choose_corrected_path,
    run_central_kmeans,
)


## Configuration

In [ ]:
DATASETS = [
    # "ecoli",
    "ovarian_cancer",
    # "ccRCC_proteomics",
    # "quartet_multiomics",
]

N_RUNS = 10
SEED_START = 5
SEED_STOP = 100
SEED_STEP = 9
SEEDS = list(range(SEED_START, SEED_STOP + 1, SEED_STEP))[:N_RUNS]

OUTPUT_ROOT = NOTEBOOK_DIR

# ── FeatureCloud settings ─────────────────────────────────────────────────────
APP_IMAGE       = "fc_kmeans_upd"
APP_SOURCE_DIR  = NOTEBOOK_DIR.parent / "federated_kmeans_upd"
CONTROLLER_HOST = "http://localhost:8000"
QUERY_INTERVAL  = 5
TIMEOUT         = 1800  # seconds per test (30 min)

# ── Publication-quality plot constants (shared with notebook 04) ──────────────
PUB_RC = {
    "font.family":       "sans-serif",
    "font.size":         9,
    "axes.titlesize":    10,
    "axes.labelsize":    9,
    "xtick.labelsize":   8.5,
    "ytick.labelsize":   8.5,
    "legend.fontsize":   8,
    "legend.framealpha": 0.9,
    "legend.edgecolor":  "#cccccc",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "axes.grid.axis":    "y",
    "grid.alpha":        0.3,
    "grid.linewidth":    0.6,
    "grid.color":        "#bbbbbb",
    "figure.dpi":        100,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
}

METHODS_ORDER = ["BC_Cntrl", "AC_Cntrl", "BC_Fed", "AC_Fed"]
METHOD_META = {
    "BC_Cntrl": dict(label="Before — Central",   color="#92c5de"),
    "AC_Cntrl": dict(label="After — Central",    color="#f4a582"),
    "BC_Fed":   dict(label="Before — Federated", color="#0571b0"),
    "AC_Fed":   dict(label="After — Federated",  color="#ca0020"),
}
DATASET_LABELS = {
    "ecoli":            "E. coli",
    "ovarian_cancer":            "Ovarian cancer",
    "ccRCC_proteomics":      "ccRCC",
    "quartet_multiomics":      "Quartet Multiomics",
}
TARGET_LABELS = {"condition": "Condition"}

print(f"Datasets : {DATASETS}")
print(f"Seeds    : {SEEDS}")
print(f"FC image : {APP_IMAGE}  |  host: {CONTROLLER_HOST}")


## Helper Functions

`compute_ari_records` — given a `{k: pd.Series}` label dict, compute ARI against the
condition ground-truth labels and return a list of record dicts.


In [ ]:
def compute_ari_records(
    cluster_labels: Dict[int, pd.Series],
    metadata: pd.DataFrame,
    k_condition: int,
    k_batch: int,
    dataset_name: str,
    method: str,
    seed: int,
) -> List[dict]:
    """Compute ARI for condition targets and return list of record dicts."""
    meta_idx = metadata.set_index("file")
    records = []
    for target, k, truth_col in [
        ("condition", k_condition, "condition"),
        # Batch-label evaluation is disabled for this workflow.
        # ("batch",     k_batch,     "lab"),
    ]:
        if k not in cluster_labels:
            continue
        pred  = cluster_labels[k]
        truth = meta_idx[truth_col].reindex(pred.index)
        mask  = truth.notna() & pred.notna()
        ari   = (
            float(adjusted_rand_score(truth[mask].astype(str), pred[mask].astype(str)))
            if mask.any() else float("nan")
        )
        records.append({
            "Dataset": dataset_name,
            "Target":  target,
            "K":       k,
            "Method":  method,
            "Seed":    seed,
            "ARI":     ari,
        })
    return records


## Load Data

For each dataset load:
- **uncorrected** matrix (shared by both central and federated conditions)
- **centrally-corrected** matrix
- **federated-corrected** matrix
- prepared **metadata** (from notebook 01)

All matrices are aligned to the metadata sample order and NA rows are dropped.

In [ ]:
from evaluation_utils.real_datasets_utils import (
    align_matrix_to_metadata,
    drop_rows_with_any_na,
)

configs = dataset_configs(REPO_ROOT)
dataset_data = {}  # ds_name → dict of matrices + metadata

for ds_name in DATASETS:
    cfg = configs[ds_name]
    prepared_dir = OUTPUT_ROOT / ds_name / "prepared"

    print(f"\n{'='*60}")
    print(f"{ds_name}")
    print(f"{'='*60}")

    if not prepared_dir.exists():
        print(f"  [SKIP] prepared directory not found — run notebook 01 first.")
        continue

    # Metadata from notebook 01
    metadata = pd.read_csv(prepared_dir / "metadata.tsv", sep="\t")
    k_condition = int(metadata["condition"].nunique())
    k_batch = int(metadata["lab"].nunique())
    k_values = [k_condition]
    # Batch-cluster k-means is disabled for this workflow.
    # k_values = sorted({k_condition, k_batch})

    # Uncorrected (before) matrix
    before_raw = load_matrix_with_lfs_fallback(cfg.before_matrix, f"{ds_name} before", cfg)
    before = align_matrix_to_metadata(before_raw, metadata, f"{ds_name} before")
    before = drop_rows_with_any_na(before, f"{ds_name} before")

    # Centrally-corrected matrix
    try:
        central_path, _ = choose_corrected_path(cfg, "central")
        central_raw = load_matrix_with_lfs_fallback(central_path, f"{ds_name} central", cfg)
        central_corr = align_matrix_to_metadata(central_raw, metadata, f"{ds_name} central")
        central_corr = drop_rows_with_any_na(central_corr, f"{ds_name} central")
    except FileNotFoundError as e:
        print(f"  [WARN] Central corrected not found: {e}")
        central_corr = None

    # Federated-corrected matrix
    try:
        fed_path, _ = choose_corrected_path(cfg, "federated")
        fed_raw = load_matrix_with_lfs_fallback(fed_path, f"{ds_name} federated", cfg)
        fed_corr = align_matrix_to_metadata(fed_raw, metadata, f"{ds_name} federated")
        fed_corr = drop_rows_with_any_na(fed_corr, f"{ds_name} federated")
    except FileNotFoundError as e:
        print(f"  [WARN] Federated corrected not found: {e}")
        fed_corr = None

    print(f"  before       : {before.shape}")
    print(f"  central_corr : {central_corr.shape if central_corr is not None else 'N/A'}")
    print(f"  fed_corr     : {fed_corr.shape if fed_corr is not None else 'N/A'}")
    print(f"  k_values     : {k_values}  (condition={k_condition}; batch={k_batch} not run)")

    dataset_data[ds_name] = {
        "before":       before,
        "central_corr": central_corr,
        "fed_corr":     fed_corr,
        "metadata":     metadata,
        "k_condition":  k_condition,
        "k_batch":      k_batch,
        "k_values":     k_values,
        "n_init":       cfg.n_init,
    }

print(f"\nDatasets loaded: {list(dataset_data.keys())}")

## Federated K-Means on Federated-Corrected Data

For each dataset and each of the 10 seeds:
1. Build per-site `inputs/corrected_fed/<site>/` directories from the federated-corrected matrix (idempotent).
2. Write all per-site `config_kmeans.yml` files with `seed: <seed>` (the `fc_kmeans_upd` app reads this parameter).
3. Launch one FeatureCloud Docker test (`fc_kmeans_upd`).
4. Read the aggregated cluster TSV and compute ARI.

**Requires:** Docker running + FeatureCloud controller at `CONTROLLER_HOST`.


In [ ]:
from evaluation_utils.real_datasets_utils import (
    write_kmeans_config,
    discover_clients,
    prepare_variant_inputs,
)
from evaluation_utils.featurecloud_kmeans_utils import run_single_federated_variant

ac_fed_records = []

for ds_name, data in dataset_data.items():
    if data["fed_corr"] is None:
        print(f"[{ds_name}] AC_Fed: federated-corrected matrix not available — skipping.")
        continue

    cfg          = configs[ds_name]
    clients      = discover_clients(cfg)
    client_names = [c.name for c in clients]
    k_values     = data["k_values"]
    ds_dir       = OUTPUT_ROOT / ds_name

    # Prepare per-site input directories ONCE from the federated-corrected matrix.
    # Uses a dedicated variant name so notebook 03's "corrected" dir is untouched.
    variant_input_dir = prepare_variant_inputs(
        dataset_root=ds_dir,
        variant_name="corrected_fed",
        matrix=data["fed_corr"],
        metadata=data["metadata"],
        clients=clients,
        k_values=k_values,
    )
    print(f"\n[{ds_name}] AC_Fed — {N_RUNS} seeds  (inputs: {variant_input_dir})")

    for seed in SEEDS:
        print(f"  seed {seed:2d} ...", end="", flush=True)

        # Inject seed into every per-site config file
        for client_name in client_names:
            write_kmeans_config(
                variant_input_dir / client_name / "config_kmeans.yml",
                k_values,
                seed=seed,
            )

        # Launch one FeatureCloud test; results saved to a fixed TSV (overwritten per seed)
        output_tsv = run_single_federated_variant(
            dataset_name=ds_name,
            variant_label="corrected_fed",
            variant_input_dir=variant_input_dir,
            dataset_root=ds_dir,
            metadata=data["metadata"],
            client_names=client_names,
            k_values=k_values,
            app_image=APP_IMAGE,
            controller_host=CONTROLLER_HOST,
            query_interval=QUERY_INTERVAL,
            timeout=TIMEOUT,
            keep_extracted=False,
            aggregate_only=False,
        )

        # Read cluster labels and compute ARI immediately
        fed_meta = pd.read_csv(output_tsv, sep="\t").set_index("file")
        labels = {
            k: fed_meta[f"Fed_{k}clusters"]
            for k in k_values
            if f"Fed_{k}clusters" in fed_meta.columns
        }
        ac_fed_records.extend(
            compute_ari_records(
                labels, data["metadata"],
                data["k_condition"], data["k_batch"],
                ds_name, "AC_Fed", seed,
            )
        )
        print(" done")

    print(f"[{ds_name}] AC_Fed complete.")

df_ac_fed = pd.DataFrame(ac_fed_records)
print(f"\nAC_Fed records: {len(df_ac_fed)}")
df_ac_fed.head()


## Federated K-Means on Uncorrected Data

Same FeatureCloud approach as above but using the uncorrected (before) matrices as per-site input.
Uses variant name `"before_multi"` to keep these inputs separate from notebook 03's `"before"` directory.


In [ ]:
bc_fed_records = []

In [ ]:
# Build a fast lookup of already finished (dataset, seed) pairs
done_pairs = {
    (r.get("Dataset"), r.get("Seed"))
    for r in bc_fed_records
    if isinstance(r, dict)
}

for ds_name, data in dataset_data.items():
    cfg          = configs[ds_name]
    clients      = discover_clients(cfg)
    client_names = [c.name for c in clients]
    k_values     = data["k_values"]
    ds_dir       = OUTPUT_ROOT / ds_name

    # Prepare per-site input directories ONCE from the uncorrected matrix
    variant_input_dir = prepare_variant_inputs(
        dataset_root=ds_dir,
        variant_name="before_multi",
        matrix=data["before"],
        metadata=data["metadata"],
        clients=clients,
        k_values=k_values,
    )
    print(f"\n[{ds_name}] BC_Fed — {N_RUNS} seeds  (inputs: {variant_input_dir})")

    for seed in SEEDS:
        print(f"  seed {seed:2d} ...", end="", flush=True)

        # skip if this dataset+seed was already computed
        if (ds_name, seed) in done_pairs:
            print(" already done")
            continue

        # Inject seed into every per-site config file
        for client_name in client_names:
            write_kmeans_config(
                variant_input_dir / client_name / "config_kmeans.yml",
                k_values,
                seed=seed,
            )

        # Launch one FeatureCloud test
        output_tsv = run_single_federated_variant(
            dataset_name=ds_name,
            variant_label="before_multi",
            variant_input_dir=variant_input_dir,
            dataset_root=ds_dir,
            metadata=data["metadata"],
            client_names=client_names,
            k_values=k_values,
            app_image=APP_IMAGE,
            controller_host=CONTROLLER_HOST,
            query_interval=QUERY_INTERVAL,
            timeout=TIMEOUT,
            keep_extracted=False,
            aggregate_only=False,
        )

        # Read cluster labels and compute ARI immediately
        fed_meta = pd.read_csv(output_tsv, sep="\t").set_index("file")
        labels = {
            k: fed_meta[f"Fed_{k}clusters"]
            for k in k_values
            if f"Fed_{k}clusters" in fed_meta.columns
        }
        bc_fed_records.extend(
            compute_ari_records(
                labels, data["metadata"],
                data["k_condition"], data["k_batch"],
                ds_name, "BC_Fed", seed,
            )
        )
        print(" done")

    print(f"[{ds_name}] BC_Fed complete.")

df_bc_fed = pd.DataFrame(bc_fed_records)
print(f"\nBC_Fed records: {len(df_bc_fed)}")
df_bc_fed.head()


## Central K-Means on Centrally-Corrected Data

Uses `run_central_kmeans` (StandardScaler + sklearn KMeans), 10 seeds.

In [ ]:
ac_cntrl_records = []

for ds_name, data in dataset_data.items():
    if data["central_corr"] is None:
        print(f"[{ds_name}] AC_Cntrl: centrally-corrected matrix not available — skipping.")
        continue
    print(f"[{ds_name}] AC_Cntrl — {N_RUNS} seeds", end="", flush=True)
    for seed in SEEDS:
        labels = run_central_kmeans(data["central_corr"], data["k_values"], seed, n_init=data["n_init"])
        ac_cntrl_records.extend(
            compute_ari_records(
                labels, data["metadata"],
                data["k_condition"], data["k_batch"],
                ds_name, "AC_Cntrl", seed,
            )
        )
        print(".", end="", flush=True)
    print()

df_ac_cntrl = pd.DataFrame(ac_cntrl_records)
print(f"\nAC_Cntrl records: {len(df_ac_cntrl)}")
df_ac_cntrl.head()

## Central K-Means on Uncorrected Data

In [ ]:
bc_cntrl_records = []

for ds_name, data in dataset_data.items():
    print(f"[{ds_name}] BC_Cntrl — {N_RUNS} seeds", end="", flush=True)
    for seed in SEEDS:
        labels = run_central_kmeans(data["before"], data["k_values"], seed, n_init=data["n_init"])
        bc_cntrl_records.extend(
            compute_ari_records(
                labels, data["metadata"],
                data["k_condition"], data["k_batch"],
                ds_name, "BC_Cntrl", seed,
            )
        )
        print(".", end="", flush=True)
    print()

df_bc_cntrl = pd.DataFrame(bc_cntrl_records)
print(f"\nBC_Cntrl records: {len(df_bc_cntrl)}")
df_bc_cntrl.head()

## Evaluation

Combine all runs, compute a summary ARI table (mean ± std across seeds),
and report boxplots showing the distribution over 10 seeds.

### Combine All Results

In [ ]:
all_parts = [df for df in [df_bc_cntrl, df_ac_cntrl, df_bc_fed, df_ac_fed] if not df.empty]
results_df = pd.concat(all_parts, ignore_index=True)

# Save full results for later use
results_path = OUTPUT_ROOT / "metrics_ari_multirun.tsv"
results_df.to_csv(results_path, sep="\t", index=False)
print(f"Full results saved to {results_path}")
print(f"Total records: {len(results_df)}")
print(results_df[["Dataset", "Target", "Method", "Seed", "ARI"]].head(10))

In [ ]:
# read back to confirm
results_path = OUTPUT_ROOT / "metrics_ari_multirun.tsv"
results_df = pd.read_csv(results_path, sep="\t")
print("\nAll done.")
print(f"Total records: {len(results_df)}")
print(results_df[["Dataset", "Target", "Method", "Seed", "ARI"]].head(10))

### ARI Summary Table

Mean ± standard deviation of ARI over 10 seeds, per dataset × target × method.
Similar layout to the table in notebook 04.

In [ ]:
def display_styled_or_plain(df, make_styler):
    try:
        display(make_styler(df))
    except AttributeError as exc:
        if "requires jinja2" not in str(exc):
            raise
        display(df)


summary_rows = []
for ds_name in DATASETS:
    ds_df = results_df[results_df["Dataset"] == ds_name]
    if ds_df.empty:
        continue
    for target in ["condition"]:
        # Batch-target summaries are disabled for this workflow.
        # for target in ["condition", "batch"]:
        tdf = ds_df[ds_df["Target"] == target]
        for method in METHODS_ORDER:
            mdf = tdf[tdf["Method"] == method]
            if mdf.empty:
                continue
            ari_vals = mdf["ARI"].dropna()
            summary_rows.append({
                "Dataset":      DATASET_LABELS.get(ds_name, ds_name),
                "Target":       TARGET_LABELS[target],
                "K":            int(mdf["K"].iloc[0]),
                "Method":       METHOD_META[method]["label"],
                "Mean ARI":     round(float(ari_vals.mean()), 4),
                "Std ARI":      round(float(ari_vals.std()), 4),
                "Min ARI":      round(float(ari_vals.min()), 4),
                "Max ARI":      round(float(ari_vals.max()), 4),
                "N runs":       int(len(ari_vals)),
            })

summary_df = pd.DataFrame(summary_rows)
display_styled_or_plain(
    summary_df,
    lambda df: (
        df.style
        .format({"Mean ARI": "{:.4f}", "Std ARI": "{:.4f}",
                 "Min ARI": "{:.4f}", "Max ARI": "{:.4f}"})
        .background_gradient(subset=["Mean ARI"], cmap="RdYlGn", vmin=0, vmax=1)
        .background_gradient(subset=["Std ARI"],  cmap="Reds",   vmin=0, vmax=0.15)
        .set_table_styles([
            {"selector": "thead th",
             "props": [("font-weight", "bold"), ("font-size", "10px"),
                       ("background-color", "#f5f5f5")]},
            {"selector": "td", "props": [("font-size", "10px"), ("padding", "4px 8px")]},
        ])
    ),
)

### ARI Boxplots

Distribution of ARI over 10 seeds for each method, dataset, and target.
Condition target only; batch-target k-means is disabled.
Each dataset group contains four boxes (BC_Cntrl, AC_Cntrl, BC_Fed, AC_Fed).

In [ ]:
from matplotlib.patches import Patch


def plot_ari_boxplots(results_df, datasets, save_path=None):
    """Publication-quality ARI boxplot: one panel per enabled target, datasets on x-axis."""
    df = results_df.copy()
    present_ds = [d for d in datasets if d in df["Dataset"].values]
    present_methods = [m for m in METHODS_ORDER if m in df["Method"].values]
    enabled_targets = [target for target in ["condition"] if target in df["Target"].values]
    # Batch-target plotting is disabled for this workflow.
    # enabled_targets = [target for target in ["condition", "batch"] if target in df["Target"].values]
    n_m = len(present_methods)
    box_w = 0.14
    group_gap = 0.20
    group_w = n_m * box_w + group_gap

    with mpl.rc_context(PUB_RC):
        fig_w = max(7.0, len(present_ds) * group_w * 4 + 2.5)
        fig, axes = plt.subplots(
            1, len(enabled_targets),
            figsize=(min(fig_w, 14), 5.0),
            sharey=True,
            constrained_layout=True,
        )

        if len(enabled_targets) == 1:
            axes = [axes]

        for ax_idx, target in enumerate(enabled_targets):
            ax = axes[ax_idx]
            tdf = df[df["Target"] == target]
            xtick_pos, xtick_labels = [], []

            for ds_idx, ds_name in enumerate(present_ds):
                ddf = tdf[tdf["Dataset"] == ds_name]
                x0 = ds_idx * group_w

                for m_idx, method in enumerate(present_methods):
                    mdf = ddf[ddf["Method"] == method]["ARI"].dropna()
                    if mdf.empty:
                        continue
                    xpos = x0 + m_idx * box_w
                    color = METHOD_META[method]["color"]

                    bp = ax.boxplot(
                        mdf.values,
                        positions=[xpos],
                        widths=box_w * 0.80,
                        patch_artist=True,
                        notch=False,
                        showfliers=True,
                        boxprops=dict(facecolor=color, color="#555555", linewidth=0.8),
                        medianprops=dict(color="#222222", linewidth=1.5),
                        whiskerprops=dict(color="#555555", linewidth=0.8),
                        capprops=dict(color="#555555", linewidth=0.8),
                        flierprops=dict(
                            marker="o", markerfacecolor=color,
                            markersize=3, alpha=0.6, markeredgewidth=0,
                        ),
                        zorder=3,
                        manage_ticks=False,
                    )

                center = x0 + (n_m - 1) * box_w / 2
                xtick_pos.append(center)
                xtick_labels.append(DATASET_LABELS.get(ds_name, ds_name))

                if ds_idx < len(present_ds) - 1:
                    ax.axvline(
                        x0 + n_m * box_w + group_gap / 2,
                        color="#aaaaaa", lw=0.9, ls="--", alpha=0.7, zorder=2,
                    )

            ax.axhline(0, color="black", lw=0.6, alpha=0.35, zorder=1)
            ax.set_ylim(-0.35, 1.12)
            ax.set_xticks(xtick_pos)
            ax.set_xticklabels(xtick_labels, fontsize=8.5)
            ax.tick_params(axis="x", length=0, pad=5)
            ax.set_title(TARGET_LABELS[target], fontweight="bold", pad=6)
            if ax_idx == 0:
                ax.set_ylabel("Adjusted Rand Index (ARI)")

        legend_handles = [
            Patch(facecolor=METHOD_META[m]["color"], edgecolor="#555555",
                  label=METHOD_META[m]["label"])
            for m in present_methods
        ]
        fig.legend(
            handles=legend_handles,
            loc="lower center", ncol=len(present_methods),
            bbox_to_anchor=(0.5, -0.10),
            framealpha=0.95, edgecolor="#cccccc",
            title=f"Method  (n={N_RUNS} seeds per box)", title_fontsize=8.5,
        )
        fig.suptitle(
            f"K-Means Clustering ARI Across {N_RUNS} Seeds: Before and After Batch Correction",
            fontweight="bold", fontsize=11,
        )
        if save_path:
            fig.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.show()
    return fig


fig_box = plot_ari_boxplots(results_df, DATASETS)

**Figure.** Boxplots of ARI across 10 random seeds for each clustering condition.
Each box shows the median (centre line), IQR (box), 1.5×IQR whiskers, and outliers (dots).
Condition target only; batch-target k-means is disabled.
Colour coding: light blue = Before — Central; orange = After — Central;
dark blue = Before — Federated; dark red = After — Federated.